# Generate Synthetic Luxury Survey Data with CTGAN

This notebook trains a **CTGAN** (Conditional Tabular GAN) model on your real survey data and generates a synthetic dataset that preserves variable relationships much better than independent marginal sampling.

**Recommended environment:** [Google Colab](https://colab.research.google.com) (free, no setup needed)

## How to use this notebook

1. Open [colab.research.google.com](https://colab.research.google.com) → **File → Upload notebook** → select this `.ipynb` file
2. Run the single cell below — it will install SDV, prompt you to upload your Excel file, train CTGAN, and download `luxury_synthetic_ctgan.xlsx` when done
3. Send the downloaded file back for evaluation

In [ ]:
# ============================================================
# Generate Synthetic Luxury Survey Data with CTGAN — single cell
# Recommended environment: Google Colab
# ============================================================

# --- Step 1: Install SDV ---
!pip install sdv --quiet
print('✓ SDV installed')

# --- Step 2: Upload your real data file ---
from google.colab import files
uploaded = files.upload()

# Get the uploaded filename automatically
filename = list(uploaded.keys())[0]
print(f'✓ Uploaded: {filename}')

# --- Step 3: Load and prepare the data ---
import pandas as pd

real = pd.read_excel(filename)

# Print all columns to help identify the correct name
print('Available columns:', real.columns.tolist())

# Convert Delivery Service text labels to numeric scale 1-5 (matches questionnaire)
delivery_map = {'Very slow': 1, 'Slow': 2, 'Average': 3, 'Fast': 4, 'Very fast': 5}
# Assuming the correct column name is 'Delivery Service Rating (1-5)' or similar.
# Please replace 'Delivery Service Rating (1-5)' below with the actual column name found in the output above.
# Example: if the column is 'Delivery Service', you would use real['Delivery Service']
if 'Q6. Delivery Service Speed' in real.columns and real['Q6. Delivery Service Speed'].dtype == object:
    real['Q6. Delivery Service Speed'] = real['Q6. Delivery Service Speed'].map(delivery_map)

print(f'Loaded {len(real)} real respondents, {len(real.columns)} columns\n')
print('Column preview:')
real.head()

# --- Step 4: Build the metadata ---
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(real)

# Force the rating columns to be treated as numerical (sometimes auto-detected as categorical)
metadata.update_column('Q5. Shopping Experience Rating', sdtype='numerical')
metadata.update_column('Q6. Delivery Service Speed',    sdtype='numerical')

print('Column types detected:')
for col, info in metadata.columns.items():
    print(f'  {col}: {info["sdtype"]}')

# --- Step 5: Train CTGAN (~2-4 minutes) ---
from sdv.single_table import CTGANSynthesizer

synthesizer = CTGANSynthesizer(
    metadata,
    epochs=300,        # increase to 500-800 for slightly better quality (slower)
   batch_size=60,  # small batch for small dataset (changed from 64 to 60)
    verbose=True,
)

synthesizer.fit(real)
print('\n✓ CTGAN training complete')

# --- Step 6: Generate synthetic respondents ---
N = 272  # number of synthetic rows (match your previous synthetic dataset size)

synthetic = synthesizer.sample(num_rows=N)

print(f'✓ Generated {len(synthetic)} synthetic respondents\n')
synthetic.head()

# --- Step 7: Quick sanity check & download ---
for c in real.columns[:4]:
    print(f'\n── {c} ──')
    print(f'  Real:      {real[c].value_counts(normalize=True).round(2).to_dict()}')
    print(f'  Synthetic: {synthetic[c].value_counts(normalize=True).round(2).to_dict()}')

# Save and download
out_file = 'luxury_synthetic_ctgan.xlsx'
synthetic.to_excel(out_file, index=False)
print(f'\n✓ Saved → {out_file}')

from google.colab import files
files.download(out_file)
